# Pixel Art via Redução de Cores (K-means) e Downsampling

**Disciplina:** Computação Científica e Análise de Dados

## Motivação e Modelagem

Transformar uma imagem em *pixel art* envolve resolver dois problemas de natureza distinta:

1. **Redução de resolução (downsampling):** dividir a imagem original em blocos e representar cada bloco por um único "pixel grande" (a média das cores do bloco).
2. **Redução de paleta de cores (quantização de cor):** limitar a imagem a apenas `k` cores, escolhidas de forma que representem bem todas as cores originais.

O segundo problema é exatamente um problema de **clusterização**: cada pixel é um ponto em $\mathbb{R}^3$ (canais R, G, B), e queremos agrupá-los em $k$ clusters que minimizem a soma dos quadrados das distâncias de cada ponto ao centro do seu cluster — ou seja, um problema de **Reconhecimento de Padrões** resolvido via **K-means**, conforme a Seção 2.7.6.2 do CoCADa.

Este notebook será construído em etapas:

1. Setup e imagem de teste
2. Downsampling (blocos → pixels grandes)
3. K-means na paleta de cores
4. Escolha de k (método do cotovelo)
5. Reconstrução e comparação visual
6. (Bônus) SVD para compressão por posto baixo


## Etapa 1 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans

%matplotlib inline
np.random.seed(42)

### Imagem de teste

Por enquanto, vamos usar uma imagem **sintética** (formas geométricas com cores conhecidas) só para
validar o pipeline. Depois você troca por uma foto real sua — basta substituir a função `carregar_imagem`
por um `Image.open('sua_foto.jpg')`.

Usar uma imagem sintética no início é uma boa prática: como sabemos exatamente quais cores existem,
conseguimos verificar se o K-means está encontrando os clusters certos.

In [ ]:
def gerar_imagem_sintetica(tamanho=240):
    """Gera uma imagem de teste com blocos de cores conhecidas + um degradê,
    para termos controle sobre o que o K-means deveria encontrar."""
    img = np.zeros((tamanho, tamanho, 3), dtype=np.uint8)

    # quatro blocos de cor sólida
    cores = [
        (230, 60, 60),    # vermelho
        (60, 140, 230),   # azul
        (60, 200, 100),   # verde
        (240, 200, 40),   # amarelo
    ]
    meio = tamanho // 2
    img[:meio, :meio] = cores[0]
    img[:meio, meio:] = cores[1]
    img[meio:, :meio] = cores[2]
    img[meio:, meio:] = cores[3]

    # um degradê diagonal por cima, pra criar variação de cor (ruído "real")
    for i in range(tamanho):
        for j in range(tamanho):
            fator = (i + j) / (2 * tamanho)
            ruido = (np.random.randn(3) * 10)
            img[i, j] = np.clip(img[i, j] * (1 - 0.3 * fator) + ruido, 0, 255)

    return Image.fromarray(img)

imagem = gerar_imagem_sintetica()
imagem_array = np.array(imagem)

plt.figure(figsize=(4, 4))
plt.imshow(imagem)
plt.title(f"Imagem de teste — shape {imagem_array.shape}")
plt.axis('off')
plt.show()

print("Dimensões:", imagem_array.shape, "  (altura, largura, canais RGB)")

---
### ✅ Checkpoint da Etapa 1

Se a célula acima rodou e mostrou uma imagem com 4 blocos coloridos e um leve degradê, está tudo certo.

**Próxima etapa (2):** vamos implementar o downsampling — dividir a imagem em blocos NxN e substituir
cada bloco pela média das suas cores, criando o efeito de "pixel grande".